In [2]:
import os

In [3]:
%pwd

'c:\\Projects\\Kidney-Disease-Classification\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Projects\\Kidney-Disease-Classification'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list
    trained_URL: str

    

In [56]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [57]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            trained_URL = training.trained_URL,
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [ ]:
import os
from cnnClassifier import logger
import gdown

In [61]:
class Training:

    def __init__(self, config: TrainingConfig):
        self.config = config


    def download_file(self):
        '''
        Download file from Google Drive using gdown library.
        '''

        try:
            trained_url = self.config.trained_URL
            trained_model_path = self.config.trained_model_path
            os.makedirs("artifacts/training", exist_ok=True)
            logger.info(f"Downloading file from {trained_url} to {trained_model_path}")

            file_id = trained_url.split("/")[-2]
            prefix = "https://drive.google.com/uc?/export=download&id="
            gdown.download(prefix + file_id, str(trained_model_path), quiet=False)

            logger.info(f"File downloaded from {trained_url} to {trained_model_path}")
        except Exception as e:
            raise e
    
    

In [62]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.download_file()
    
except Exception as e:
    raise e

[2026-07-18 21:24:12,414: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-18 21:24:12,421: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-18 21:24:12,426: INFO: common: created directory at: artifacts]
[2026-07-18 21:24:12,426: INFO: common: created directory at: artifacts\training]
[2026-07-18 21:24:12,426: INFO: 1259205682: Downloading file from https://drive.google.com/file/d/16kDEQXA3mC093nuGXLKZFdgOzHSM9_w8/view?usp=sharing to artifacts\training\trained_model.keras]


Downloading...
From (original): https://drive.google.com/uc?id=16kDEQXA3mC093nuGXLKZFdgOzHSM9_w8
From (redirected): https://drive.google.com/uc?id=16kDEQXA3mC093nuGXLKZFdgOzHSM9_w8&confirm=t&uuid=f0972efd-a0f9-4ae7-b9b0-5f221c72b2fe
To: c:\Projects\Kidney-Disease-Classification\artifacts\training\trained_model.keras
100%|██████████| 46.3M/46.3M [00:13<00:00, 3.51MB/s]


[2026-07-18 21:24:29,588: INFO: 1259205682: File downloaded from https://drive.google.com/file/d/16kDEQXA3mC093nuGXLKZFdgOzHSM9_w8/view?usp=sharing to artifacts\training\trained_model.keras]
